In [4]:
!pip install stix2


In [3]:
from typing import Iterable, Dict, Any
import json
from stix2 import MemoryStore, Filter, parse  # no load_bundle in some versions

# ── loader that works across stix2 versions ───────────────────────────────────
def load_attack_store(path: str) -> MemoryStore:
    with open(path, "r") as f:
        raw = json.load(f)
    try:
        bundle = parse(raw, allow_custom=True)  # preserves x_mitre_* fields
        objs = bundle.objects
    except Exception:
        objs = raw.get("objects", raw)  # tolerate dict/list shapes
    return MemoryStore(stix_data=objs)

# ── shared helper to extract the ATT&CK external_id (TA0005, G0092, S0037, etc.) ─
def _get_attack_external_id(obj: Dict[str, Any], source_name: str = "mitre-attack"):
    for ref in obj.get("external_references", []) or []:
        if isinstance(ref, dict):
            src = ref.get("source_name")
            ext_id = ref.get("external_id")
        else:
            src = getattr(ref, "source_name", None)
            ext_id = getattr(ref, "external_id", None)
        if src == source_name and ext_id:
            return ext_id
    return None

# ── techniques ────────────────────────────────────────────────────────────────
def build_technique_name_to_id_map(techniques: Iterable[Dict[str, Any]]) -> Dict[str, str]:
    """
    {"Process Injection": "T1055", "Masquerading": "T1036", ...}
    """
    out = {}
    for t in techniques:
        name = t.get("name")
        ext_id = _get_attack_external_id(t)
        if name and ext_id:
            out[name] = ext_id
    return out

# ── tactics ───────────────────────────────────────────────────────────────────
def build_tactic_name_to_id_map(tactics: Iterable[Dict[str, Any]]) -> Dict[str, str]:
    """{"Defense Evasion": "TA0005", "Persistence": "TA0003", ...}"""
    tac_map = {}
    for t in tactics:
        name = t.get("name")
        ext_id = _get_attack_external_id(t)
        if name and ext_id:
            tac_map[name] = ext_id
    return tac_map

def build_tactic_shortname_to_id_map(tactics: Iterable[Dict[str, Any]]) -> Dict[str, str]:
    """{"defense-evasion": "TA0005", "persistence": "TA0003", ...} (x_mitre_shortname)"""
    short_map = {}
    for t in tactics:
        short = t.get("x_mitre_shortname")
        ext_id = _get_attack_external_id(t)
        if short and ext_id:
            short_map[short] = ext_id
    return short_map

# ── groups (intrusion-sets) ───────────────────────────────────────────────────
def build_group_name_to_id_map(groups: Iterable[Dict[str, Any]], include_aliases: bool = True) -> Dict[str, str]:
    """
    {"Wizard Spider": "G0102", "FIN6": "G0037", ...}
    If include_aliases=True, aliases also map to the same ID.
    """
    grp_map = {}
    for g in groups:
        name = g.get("name")
        ext_id = _get_attack_external_id(g)
        if name and ext_id:
            grp_map[name] = ext_id
        if include_aliases and ext_id:
            # cover both fields
            for alias in (g.get("aliases", []) or []) + (g.get("x_mitre_aliases", []) or []):
                if alias:
                    grp_map.setdefault(alias, ext_id)
    return grp_map

# ── malware & tools (software) ────────────────────────────────────────────────
def build_software_name_to_id_map(objs: Iterable[Dict[str, Any]], include_aliases: bool = True) -> Dict[str, str]:
    """
    Works for malware OR tool SDOs.
    {"TrickBot": "S0266", "Cobalt Strike": "S0154", ...}
    """
    sw_map = {}
    for o in objs:
        name = o.get("name")
        ext_id = _get_attack_external_id(o)
        if name and ext_id:
            sw_map[name] = ext_id
        if include_aliases and ext_id:
            for alias in (o.get("aliases", []) or []) + (o.get("x_mitre_aliases", []) or []):
                if alias:
                    sw_map.setdefault(alias, ext_id)
    return sw_map

# ── example usage ─────────────────────────────────────────────────────────────
MS = load_attack_store("attack-stix-data/enterprise-attack/enterprise-attack.json")

techniques = MS.query([Filter("type", "=", "attack-pattern")])
tactics    = MS.query([Filter("type", "=", "x-mitre-tactic")])
groups     = MS.query([Filter("type", "=", "intrusion-set")])
malware    = MS.query([Filter("type", "=", "malware")])
tools      = MS.query([Filter("type", "=", "tool")])

tech_name_to_id    = build_technique_name_to_id_map(techniques)
tactic_name_to_id  = build_tactic_name_to_id_map(tactics)
tactic_short_to_id = build_tactic_shortname_to_id_map(tactics)
group_name_to_id   = build_group_name_to_id_map(groups, include_aliases=True)
malware_name_to_id = build_software_name_to_id_map(malware, include_aliases=True)
tool_name_to_id    = build_software_name_to_id_map(tools, include_aliases=True)

# Unified software map:
software_name_to_id = {**malware_name_to_id, **tool_name_to_id}

# quick sanity checks:
print("techniques:", len(tech_name_to_id))
print("tactics:", len(tactic_name_to_id))
print("groups:", len(group_name_to_id))
print("software:", len(software_name_to_id))


techniques: 696
tactics: 14
groups: 550
software: 1032


In [13]:
groups[0]

IntrusionSet(type='intrusion-set', spec_version='2.1', id='intrusion-set--01e28736-2ffc-455b-9880-ed4d1407ae07', created_by_ref='identity--c78cb6e5-0c4b-4611-8297-d1b8b55e40b5', created='2021-01-06T17:46:35.134Z', modified='2024-10-28T19:11:56.485Z', name='Indrik Spider', description='[Indrik Spider](https://attack.mitre.org/groups/G0119) is a Russia-based cybercriminal group that has been active since at least 2014. [Indrik Spider](https://attack.mitre.org/groups/G0119) initially started with the [Dridex](https://attack.mitre.org/software/S0384) banking Trojan, and then by 2017 they began running ransomware operations using [BitPaymer](https://attack.mitre.org/software/S0570), [WastedLocker](https://attack.mitre.org/software/S0612), and Hades ransomware. Following U.S. sanctions and an indictment in 2019, [Indrik Spider](https://attack.mitre.org/groups/G0119) changed their tactics and diversified their toolset.(Citation: Crowdstrike Indrik November 2018)(Citation: Crowdstrike EvilCorp

In [8]:
def build_group_name_to_id_and_aliases_csv(groups, include_self=False):
    """
    {"Indrik Spider": {"id":"G0119","aliases_csv":"DEV-0243, Evil Corp, Manatee Tempest, UNC2165"}, ...}
    """
    out = {}
    for g in groups:
        gid = _get_attack_external_id(g)
        name = g.get("name")
        if not (gid and name):
            continue

        aliases = set()
        for a in (g.get("aliases", []) or []):
            if a: aliases.add(a)
        for a in (g.get("x_mitre_aliases", []) or []):
            if a: aliases.add(a)
        if not include_self:
            aliases.discard(name)

        out[name] = {
            "id": gid,
            "aliases_csv": ", ".join(sorted(aliases))
        }
    return out

build_group_name_to_id_and_aliases_csv(groups, False)

{'Indrik Spider': {'id': 'G0119',
  'aliases_csv': 'DEV-0243, Evil Corp, Manatee Tempest, UNC2165'},
 'LuminousMoth': {'id': 'G1014', 'aliases_csv': ''},
 'Wizard Spider': {'id': 'G0102',
  'aliases_csv': 'DEV-0193, FIN12, GOLD BLACKBURN, Grim Spider, ITG23, Periwinkle Tempest, TEMP.MixMaster, UNC1878'},
 'Elderwood': {'id': 'G0066',
  'aliases_csv': 'Beijing Group, Elderwood Gang, Sneaky Panda'},
 'Frankenstein': {'id': 'G0101', 'aliases_csv': ''},
 'FIN7': {'id': 'G0046',
  'aliases_csv': 'Carbon Spider, ELBRUS, GOLD NIAGARA, ITG14, Sangria Tempest'},
 'Velvet Ant': {'id': 'G1047', 'aliases_csv': ''},
 'WIRTE': {'id': 'G0090', 'aliases_csv': ''},
 'Dragonfly': {'id': 'G0035',
  'aliases_csv': 'BROMINE, Berserk Bear, Crouching Yeti, DYMALLOY, Energetic Bear, Ghost Blizzard, IRON LIBERTY, TEMP.Isotope, TG-4192'},
 'OilRig': {'id': 'G0049',
  'aliases_csv': 'APT34, COBALT GYPSY, Crambus, EUROPIUM, Earth Simnavaz, Evasive Serpens, Hazel Sandstorm, Helix Kitten, IRN2, ITG13, TA452'},
 'Eq

In [9]:
def parse_intrusion_set(g):
    """
    Flatten a single STIX IntrusionSet into a clean dict.
    """
    # pull ATT&CK G-ID + ATT&CK URL from external_references
    g_id, g_url = None, None
    for ref in g.get("external_references", []) or []:
        src = getattr(ref, "source_name", None) if not isinstance(ref, dict) else ref.get("source_name")
        if src == "mitre-attack":
            g_id  = getattr(ref, "external_id", None) if not isinstance(ref, dict) else ref.get("external_id")
            g_url = getattr(ref, "url", None)         if not isinstance(ref, dict) else ref.get("url")
            break

    # “Associated Groups” on the website = aliases/x_mitre_aliases
    aliases = set()
    for a in (g.get("aliases", []) or []):
        if a: aliases.add(a)
    for a in (g.get("x_mitre_aliases", []) or []):
        if a: aliases.add(a)

    return {
        "stix_ref": g.get("id"),
        "group_id": g_id,                   # e.g., "G0119"
        "attack_url": g_url,                # e.g., https://attack.mitre.org/groups/G0119
        "name": g.get("name"),              # e.g., "Indrik Spider"
        "aliases": sorted(aliases),         # e.g., ["Evil Corp","Manatee Tempest","DEV-0243","UNC2165","Indrik Spider"]
        "description": g.get("description"),
        "domains": g.get("x_mitre_domains", []),
        "created": g.get("created"),
        "modified": g.get("modified"),
        "revoked": g.get("revoked", False),
    }

def label_matches_intrusion_set(label: str, g) -> bool:
    """
    True if a free-text label (e.g., 'Evil Corp', 'UNC2165') matches this group by
    canonical name or any alias.
    """
    lab = (label or "").strip()
    if not lab:
        return False
    if lab == g.get("name"):
        return True
    if lab in (g.get("aliases", []) or []):
        return True
    if lab in (g.get("x_mitre_aliases", []) or []):
        return True
    return False


In [10]:
g = groups[0]  # the Indrik Spider object you printed
rec = parse_intrusion_set(g)
rec

{'stix_ref': 'intrusion-set--01e28736-2ffc-455b-9880-ed4d1407ae07',
 'group_id': 'G0119',
 'attack_url': 'https://attack.mitre.org/groups/G0119',
 'name': 'Indrik Spider',
 'aliases': ['DEV-0243',
  'Evil Corp',
  'Indrik Spider',
  'Manatee Tempest',
  'UNC2165'],
 'description': '[Indrik Spider](https://attack.mitre.org/groups/G0119) is a Russia-based cybercriminal group that has been active since at least 2014. [Indrik Spider](https://attack.mitre.org/groups/G0119) initially started with the [Dridex](https://attack.mitre.org/software/S0384) banking Trojan, and then by 2017 they began running ransomware operations using [BitPaymer](https://attack.mitre.org/software/S0570), [WastedLocker](https://attack.mitre.org/software/S0612), and Hades ransomware. Following U.S. sanctions and an indictment in 2019, [Indrik Spider](https://attack.mitre.org/groups/G0119) changed their tactics and diversified their toolset.(Citation: Crowdstrike Indrik November 2018)(Citation: Crowdstrike EvilCorp Ma

In [19]:
# -*- coding: utf-8 -*-
"""
Resolve an Intrusion Set's techniques + tactics from a local ATT&CK Enterprise STIX dump.
Works across stix2 versions (no need for load_bundle).
"""

from typing import Iterable, Dict, Any, List, Optional, Tuple
import json
from collections import defaultdict

from stix2 import MemoryStore, FileSystemStore, Filter, parse

# ──────────────────────────────────────────────────────────────────────────────
# Loaders
# ──────────────────────────────────────────────────────────────────────────────

def load_attack_store(path_or_dir: str) -> MemoryStore:
    """
    If given a directory containing ATT&CK STIX JSONs, use FileSystemStore.
    If given a single JSON file, parse and load into MemoryStore.
    """
    try:
        # crude check: if it looks like a directory, try FileSystemStore
        import os
        if os.path.isdir(path_or_dir):
            return FileSystemStore(path_or_dir)  # queryable like MemoryStore
        # else assume it's a file
        with open(path_or_dir, "r") as f:
            raw = json.load(f)
        try:
            bundle = parse(raw, allow_custom=True)  # preserves x_mitre_* fields
            objs = bundle.objects
        except Exception:
            objs = raw.get("objects", raw)  # tolerate dict/list shapes
        return MemoryStore(stix_data=objs)
    except Exception as e:
        raise RuntimeError(f"Failed to load ATT&CK data from '{path_or_dir}': {e}")

# ──────────────────────────────────────────────────────────────────────────────
# Helpers
# ──────────────────────────────────────────────────────────────────────────────

def _get_attack_external_id(obj: Dict[str, Any], source_name: str = "mitre-attack") -> Optional[str]:
    for ref in obj.get("external_references", []) or []:
        if isinstance(ref, dict):
            src = ref.get("source_name")
            ext_id = ref.get("external_id")
        else:
            src = getattr(ref, "source_name", None)
            ext_id = getattr(ref, "external_id", None)
        if src == source_name and ext_id:
            return ext_id
    return None

def build_tactic_shortname_to_id_map(tactics: Iterable[Dict[str, Any]]) -> Dict[str, str]:
    """
    {"defense-evasion": "TA0005", "persistence": "TA0003", ...}
    """
    out = {}
    for t in tactics:
        short = t.get("x_mitre_shortname")
        ext_id = _get_attack_external_id(t)
        if short and ext_id:
            out[short] = ext_id
    return out

def technique_rows_from_attack_pattern(ap: Dict[str, Any]) -> Dict[str, Any]:
    tid  = _get_attack_external_id(ap)
    name = ap.get("name")
    tacs_short = []
    for ph in (ap.get("kill_chain_phases") or []):
        if ph.get("kill_chain_name") == "mitre-attack":
            short = ph.get("phase_name")
            if short:
                tacs_short.append(short)
    return {
        "technique_id": tid,
        "technique_name": name,
        "tactic_shortnames": sorted(set(tacs_short)),
    }

def _is_intrusion_set(obj: Dict[str, Any]) -> bool:
    return (obj.get("type") == "intrusion-set")

def _name_and_aliases(obj: Dict[str, Any]) -> List[str]:
    names = set()
    n = obj.get("name")
    if n: names.add(n)
    for a in (obj.get("aliases", []) or []):
        if a: names.add(a)
    for a in (obj.get("x_mitre_aliases", []) or []):
        if a: names.add(a)
    return list(names)

def resolve_intrusion_set_id(ms: MemoryStore, key: str) -> Optional[str]:
    """
    Resolve a user-provided key to a STIX intrusion-set ID.
    Key can be:
      - the STIX ID (e.g., "intrusion-set--01e2...")
      - the ATT&CK Group ID (e.g., "G0119")
      - a name or alias (e.g., "Indrik Spider", "Evil Corp")
    Returns the STIX ID or None.
    """
    key_norm = (key or "").strip()

    # 1) If it's already a STIX intrusion-set id
    if key_norm.startswith("intrusion-set--"):
        return key_norm

    # 2) Try ATT&CK external G-ID match
    if key_norm.upper().startswith("G0"):
        grps = ms.query([Filter("type", "=", "intrusion-set")])
        for g in grps:
            gid = _get_attack_external_id(g)
            if gid and gid.upper() == key_norm.upper():
                return g.get("id")

    # 3) Try name/alias match (case-insensitive)
    grps = ms.query([Filter("type", "=", "intrusion-set")])
    key_lower = key_norm.lower()
    for g in grps:
        for n in _name_and_aliases(g):
            if n.lower() == key_lower:
                return g.get("id")

    return None

# ──────────────────────────────────────────────────────────────────────────────
# Core: collect techniques (with tactics) used by an Intrusion Set
# ──────────────────────────────────────────────────────────────────────────────

def collect_group_ttps(ms: MemoryStore, group_stix_id: str) -> List[Dict[str, Any]]:
    """
    Returns a de-duplicated list of rows:
      {
        "technique_id": "T1055",
        "technique_name": "Process Injection",
        "tactic_shortnames": ["defense-evasion","privilege-escalation"],
      }
    Traverses:
      - direct: intrusion-set -> attack-pattern (relationship_type="uses")
      - two-hop: intrusion-set -> (malware|tool) -> attack-pattern
    """
    # Ensure we actually have relationships loaded
    total_rels = ms.query([Filter("type", "=", "relationship")])
    if not total_rels:
        raise ValueError("No relationship objects found in the loaded ATT&CK data. "
                         "Load a complete Enterprise ATT&CK dump (including relationships).")

    rows: List[Dict[str, Any]] = []
    seen_tid: set = set()

    # 1) Direct: group → technique
    rels = ms.query([
        Filter("type", "=", "relationship"),
        Filter("source_ref", "=", group_stix_id),
        Filter("relationship_type", "=", "uses"),
    ])

    # Also collect software targets for step 2
    software_ids: List[str] = []

    for r in rels:
        tgt = r.get("target_ref", "")
        if tgt.startswith("attack-pattern"):
            ap = ms.get(tgt)
            if ap:
                row = technique_rows_from_attack_pattern(ap)
                tid = row.get("technique_id")
                if tid and tid not in seen_tid:
                    rows.append(row)
                    seen_tid.add(tid)
        elif tgt.startswith("malware") or tgt.startswith("tool"):
            software_ids.append(tgt)

    # 2) Via software: malware/tool → technique
    for sw_id in software_ids:
        uses_rels = ms.query([
            Filter("type", "=", "relationship"),
            Filter("source_ref", "=", sw_id),
            Filter("relationship_type", "=", "uses"),
        ])
        for ur in uses_rels:
            tgt = ur.get("target_ref", "")
            if tgt.startswith("attack-pattern"):
                ap = ms.get(tgt)
                if ap:
                    row = technique_rows_from_attack_pattern(ap)
                    tid = row.get("technique_id")
                    if tid and tid not in seen_tid:
                        rows.append(row)
                        seen_tid.add(tid)

    return rows

# ──────────────────────────────────────────────────────────────────────────────
# Public API
# ──────────────────────────────────────────────────────────────────────────────

def get_intrusion_set_ttps(
    ms: MemoryStore,
    group_key: str,
    return_tactic_ids: bool = True
) -> Tuple[List[Dict[str, Any]], Dict[str, str]]:
    """
    Resolve an intrusion set and return its techniques with tactics.

    Parameters
    ----------
    ms : MemoryStore
    group_key : str
        STIX ID ("intrusion-set--..."), ATT&CK G-ID ("G0119"), or name/alias ("Indrik Spider", "Evil Corp").
    return_tactic_ids : bool
        If True, also adds "tactic_ids": ["TA0005", ...] using the tactic shortname → ID map.

    Returns
    -------
    rows : list of dicts
        Each row has keys: technique_id, technique_name, tactic_shortnames, (optional) tactic_ids
    tactic_short_to_id : dict
        Mapping of tactic shortnames to TA IDs for convenience.
    """
    group_id = resolve_intrusion_set_id(ms, group_key)
    if not group_id:
        raise ValueError(f"Could not resolve intrusion set for key: {group_key}")

    # build tactic map
    tactics = ms.query([Filter("type", "=", "x-mitre-tactic")])
    tactic_short_to_id = build_tactic_shortname_to_id_map(tactics)

    rows = collect_group_ttps(ms, group_id)

    if return_tactic_ids:
        for r in rows:
            r["tactic_ids"] = [tactic_short_to_id.get(s, s) for s in r["tactic_shortnames"]]

    # sort for stable output
    rows.sort(key=lambda x: (x["technique_id"] or "", x["technique_name"] or ""))

    return rows, tactic_short_to_id

# ──────────────────────────────────────────────────────────────────────────────
# Example usage
# ──────────────────────────────────────────────────────────────────────────────

if __name__ == "__main__":
    # 1) Point this to your ATT&CK Enterprise STIX bundle or directory
    #    e.g., "attack-stix-data/enterprise-attack/enterprise-attack.json"
    #    or a folder containing ATT&CK JSONs.
    ATTACK_PATH = "attack-stix-data/enterprise-attack/enterprise-attack.json"

    MS = load_attack_store(ATTACK_PATH)

    # 2) Query by group ATT&CK ID, name, alias, or STIX ID:
    GROUP_KEY = "G0037"          # Indrik Spider
    # GROUP_KEY = "Indrik Spider" # or alias like "Evil Corp"
    # GROUP_KEY = "intrusion-set--01e28736-2ffc-455b-9880-ed4d1407ae07"  # STIX ID

    rows, tac_map = get_intrusion_set_ttps(MS, GROUP_KEY, return_tactic_ids=True)

    print(f"\nTechniques used by {GROUP_KEY}: {len(rows)} found")
    # Print a few examples
    for r in rows[:10]:
        print(f"  {r['technique_id']:<6}  {r['technique_name']:<45}"
              f" tactics={r['tactic_shortnames']}  ids={r.get('tactic_ids')}")
?


Techniques used by G0037: 144 found
  T1001   Data Obfuscation                              tactics=['command-and-control']  ids=['TA0011']
  T1001.001  Junk Data                                     tactics=['command-and-control']  ids=['TA0011']
  T1001.003  Protocol or Service Impersonation             tactics=['command-and-control']  ids=['TA0011']
  T1003.001  LSASS Memory                                  tactics=['credential-access']  ids=['TA0006']
  T1003.002  Security Account Manager                      tactics=['credential-access']  ids=['TA0006']
  T1003.003  NTDS                                          tactics=['credential-access']  ids=['TA0006']
  T1003.004  LSA Secrets                                   tactics=['credential-access']  ids=['TA0006']
  T1003.006  DCSync                                        tactics=['credential-access']  ids=['TA0006']
  T1005   Data from Local System                        tactics=['collection']  ids=['TA0009']
  T1007   System Service 


IPython -- An enhanced Interactive Python

IPython offers a fully compatible replacement for the standard Python
interpreter, with convenient shell features, special commands, command
history mechanism and output results caching.

At your system command line, type 'ipython -h' to see the command line
options available. This document only describes interactive features.

GETTING HELP
------------

Within IPython you have various way to access help:

  ?         -> Introduction and overview of IPython's features (this screen).
  object?   -> Details about 'object'.
  object??  -> More detailed, verbose information about 'object'.
  %quickref -> Quick reference of all IPython specific syntax and magics.
  help      -> Access Python's own help system.

If you are in terminal IPython you can quit this screen by pressing `q`.


MAIN FEATURES
-------------

* Access to the standard Python help with object docstrings and the Python
  manuals. Simply type 'help' (no quotes) to invoke it.

* Ma